# Pianalysis：Colab Pro 训练

在 Colab 的「运行时 → 更改运行时类型」中选择 GPU，然后依次运行下方三个代码单元。使用自己的 Colab Pro 账号；Pro 不保证 GPU 型号或固定会话时长。

训练包应位于 Drive 项目目录的 `inputs/` 下，文件为 `cloud_training_bundle.tar.gz` 与 `cloud_training_bundle.tar.gz.manifest.json`；如果上传的是 `.part001`、`.part002` 等分片，本页会按外部清单自动逐片校验并拼接，再校验完整包。本页先校验每个文件，再建立独立 Python 3.11 环境，不修改 Colab 自带的 PyTorch。

只需设置三个选项。`MAX_HOURS` 是训练调度的总时限，包含数据检查、短 profile、训练和同步；环境安装时间另计。达到上限会尝试保存，最多再给 60 秒终止宽限；Drive 网络延迟仍可能增加收尾时间。

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/Pianalysis"
STAGE = "texture"  # "harmony"、"texture" 或 "both"
MAX_HOURS = 2.0

from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import hashlib
import importlib.util
import json
from pathlib import Path
import shutil
import sys
import tarfile
import uuid

project_drive = Path(PROJECT_DIR)
archive_name = "cloud_training_bundle.tar.gz"
external_manifest = project_drive / "inputs" / (archive_name + ".manifest.json")
manifest = json.loads(external_manifest.read_text(encoding="utf-8"))
expected_hash = manifest["archive_sha256"]
if len(expected_hash) != 64 or any(c not in "0123456789abcdef" for c in expected_hash):
    raise ValueError("训练包 SHA256 无效")
cache = Path("/content/pianalysis_inputs")
cache.mkdir(parents=True, exist_ok=True)
archive_path = cache / (expected_hash + ".tar.gz")
def file_sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()
if not archive_path.exists():
    temporary = cache / (uuid.uuid4().hex + ".part")
    whole_archive = project_drive / "inputs" / archive_name
    with temporary.open("xb") as dst:
        if whole_archive.exists():
            with whole_archive.open("rb") as src:
                shutil.copyfileobj(src, dst, length=8 * 1024 * 1024)
        else:
            parts = manifest.get("archive_parts")
            if not isinstance(parts, list) or not parts:
                raise ValueError("未找到完整训练包，外部清单也没有 archive_parts")
            for index, part in enumerate(parts, 1):
                if not isinstance(part, dict):
                    raise ValueError("分片清单格式无效")
                name, size, checksum = part.get("name"), part.get("bytes"), part.get("sha256")
                expected_name = f"{archive_name}.part{index:03d}"
                if not isinstance(name, str) or Path(name).name != name or name != expected_name:
                    raise ValueError("分片名称必须是安全、连续编号的 basename：" + expected_name)
                if type(size) is not int or size <= 0:
                    raise ValueError("分片大小无效：" + name)
                if not isinstance(checksum, str) or len(checksum) != 64 or any(c not in "0123456789abcdef" for c in checksum):
                    raise ValueError("分片 SHA256 无效：" + name)
            if sum(part["bytes"] for part in parts) != manifest["archive_bytes"]:
                raise ValueError("分片总大小与完整包清单不符")
            for part in parts:
                part_hash, part_bytes = hashlib.sha256(), 0
                with (project_drive / "inputs" / part["name"]).open("rb") as src:
                    for block in iter(lambda: src.read(8 * 1024 * 1024), b""):
                        part_bytes += len(block)
                        if part_bytes > part["bytes"]:
                            raise ValueError("分片超过清单大小：" + part["name"])
                        part_hash.update(block)
                        dst.write(block)
                if part_bytes != part["bytes"] or part_hash.hexdigest() != part["sha256"]:
                    raise ValueError("分片未上传完整或校验失败：" + part["name"])
    if temporary.stat().st_size != manifest["archive_bytes"] or file_sha(temporary) != expected_hash:
        raise ValueError("Drive 训练包未上传完整或校验失败；未解包")
    temporary.replace(archive_path)
if archive_path.stat().st_size != manifest["archive_bytes"] or file_sha(archive_path) != expected_hash:
    raise ValueError("本地缓存校验失败")
local_manifest = cache / (expected_hash + ".manifest.json")
local_manifest.write_text(json.dumps(manifest, ensure_ascii=False), encoding="utf-8")

# 仅引导加载清单里经过 SHA 校验的固定辅助文件；后续完整解包拒绝链接、越界路径及未列出的文件。
helper_record = next(x for x in manifest["files"] if x["path"] == "scripts/colab_runtime.py")
with tarfile.open(archive_path, "r:gz") as archive:
    member = archive.getmember("scripts/colab_runtime.py")
    if not member.isfile() or member.size != helper_record["bytes"] or member.size > 1024 * 1024:
        raise ValueError("引导文件类型或大小无效")
    helper_bytes = archive.extractfile(member).read()
if hashlib.sha256(helper_bytes).hexdigest() != helper_record["sha256"]:
    raise ValueError("引导文件校验失败")
bootstrap_dir = Path("/content/pianalysis_bootstrap")
bootstrap_dir.mkdir(exist_ok=True)
helper_path = bootstrap_dir / "colab_runtime.py"
helper_path.write_bytes(helper_bytes)
spec = importlib.util.spec_from_file_location("pianalysis_colab_bootstrap", helper_path)
helper = importlib.util.module_from_spec(spec)
spec.loader.exec_module(helper)
PROJECT = Path("/content/Pianalysis")
helper.safe_extract_bundle(archive_path, local_manifest, PROJECT)
PYTHON = helper.prepare_environment(PROJECT, project_drive)
print("校验与独立环境完成：", PYTHON)


In [ ]:
import subprocess

if STAGE not in ("harmony", "texture", "both"):
    raise ValueError("STAGE 只能为 harmony、texture、both")
command = [PYTHON, str(PROJECT / "scripts/colab_train.py"),
           "--drive-root", PROJECT_DIR, "--stage", STAGE,
           "--max-hours", str(MAX_HOURS)]
subprocess.run(command, cwd=PROJECT, check=True)


## 保存与恢复

每阶段先进行 6 个真实优化步的短 profile，再继续同一个优化器和调度器；这 6 步计入训练，不是丢弃的伪运行。首次遇到 T4 会使用 FP32，并降低 microbatch、增加梯度累积以保持有效 batch。恢复时沿用原配置；若原 run 使用 BF16 而本次 GPU 不支持，会停止并要求使用兼容 GPU，不静默改变实验。

Drive 布局：

- `runs/<阶段>/run_<id>/checkpoints/g_*/`：checkpoint 与带 hash 的 `commit.json`。
- `runs/<阶段>/run_<id>/latest/`：方便查看的日志、有效配置和报告；恢复不依赖覆盖写的这些副本。
- 同一 run 的 `artifacts/objects/` 与 `artifacts/snapshots/`：所有日志、配置、profile、指标与报告的可校验版本。
- `runs/_environment/`：安装日志、依赖版本及环境说明。

默认每个 run 保留最近 **2 个 last + 1 个 best**。先上传新代、回读校验并提交，再只清理该 run 自己生成的过期 checkpoint；不会清理其他目录。日志与报告版本保留，Drive 容量仍需足够。checkpoint 上传会暂时阻塞训练，日志约每 10 秒同步。

断线后重新运行本页即可：恢复会核对数据、词表、训练代码及有效配置，并检查 checkpoint 内容 hash。最新一代损坏或没有完整提交记录时，会退回上一完整代；不会假设 Drive FUSE 的 rename 是原子事务。尚未成功提交的当前 checkpoint，以及最近一次同步之后的日志，仍可能随 Colab runtime 消失。

`/content` 中的解压暂存、下载缓存、uv 缓存和 Python 环境属于可重建缓存，不上传重复副本。模型状态、优化器/调度器、随机状态、有效配置、训练日志、数据审计、profile 和最终报告才是需要保存的运行产物。已早停或达到最大步数的 run 会保留结果，不自动继续更新参数。

这页不会自动关闭 Colab 会话。训练完成后可在 Colab 中断开并删除 runtime；Drive 中已提交的结果会保留。
